# Exercício Prático - Queries ORM

Questões retiradas de learn-python: sqlalchemy_orm-questions.

## 1) Baixar e extrair o Chinook database

In [ ]:
import urllib.request
import zipfile
import os

if not os.path.exists("chinook.db"):
    url = "http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip"
    urllib.request.urlretrieve(url, "chinook.zip")
    with zipfile.ZipFile("chinook.zip") as z:
        z.extractall(".")

print("ok")

## 2) Setup do ORM com automap

Aqui a gente conecta no banco e o SQLAlchemy monta os modelos automaticamente a partir das tabelas que já existem.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.orm import Session

engine = create_engine("sqlite:///chinook.db")
Base = automap_base()
Base.prepare(engine, reflect=True)

# olhando as tabelas disponíveis
print(Base.classes.keys())

session = Session(engine)

## 3) Operações solicitadas

### Imprima os três primeiros registros da tabela tracks

In [ ]:
Track = Base.classes.tracks

for t in session.query(Track).limit(3):
    print(t.TrackId, t.Name)

### Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks

In [ ]:
Album = Base.classes.albums

resultado = session.query(Track, Album).join(Album).limit(20).all()

for t, a in resultado:
    print(t.Name, '|', a.Title)

### Imprima as 10 primeiras vendas de faixas da tabela invoice_items

In [ ]:
InvoiceItem = Base.classes.invoice_items

for item in session.query(InvoiceItem).limit(10):
    print(item.InvoiceLineId, item.TrackId, item.Quantity, item.UnitPrice)

### Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida

In [ ]:
resultado = session.query(InvoiceItem, Track).join(Track).limit(10).all()

for item, t in resultado:
    print(t.Name, '-', item.Quantity)

### Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas

In [ ]:
from sqlalchemy import func

mais_vendidas = session.query(Track.Name, func.sum(InvoiceItem.Quantity)) \
    .join(InvoiceItem) \
    .group_by(Track.TrackId) \
    .order_by(func.sum(InvoiceItem.Quantity).desc()) \
    .limit(10).all()

for nome, qtd in mais_vendidas:
    print(nome, '-', qtd)

### Quem são os 10 artistas que mais venderam?

Dica: precisa juntar invoice_items, tracks, albums e artists.

In [ ]:
Artist = Base.classes.artists

top_artistas = session.query(Artist.Name, func.sum(InvoiceItem.Quantity)) \
    .join(Album, Album.ArtistId == Artist.ArtistId) \
    .join(Track, Track.AlbumId == Album.AlbumId) \
    .join(InvoiceItem, InvoiceItem.TrackId == Track.TrackId) \
    .group_by(Artist.ArtistId) \
    .order_by(func.sum(InvoiceItem.Quantity).desc()) \
    .limit(10).all()

for nome, qtd in top_artistas:
    print(nome, '-', qtd)